# 蒙特卡洛探索开始（MC Exploring Starts）

> **📌 First-Visit vs Every-Visit——两种回报记账法**
>
> 一条轨迹中，同一个 $(s,a)$ 可能反复出现，两种方法的区别在于：**重复出现的样本要不要？**
>
> - **Every-Visit**：不管出现几次，每次都攒进去取平均——多多益善，收敛快，但同一回合的多个样本存在相关性。
> - **First-Visit**：每个回合只认第一次，之后再出现一律忽略——样本独立性更好，代价是样本少一些。
>
> ⚠️ **易错点**：First-Visit 的"只认第一次"是**以回合为单位**重置的，新回合开始后又重新计数。它并不是说某个 $(s,a)$ 整个训练过程只能更新一次。
>
> 以下面这条轨迹为例：
> $$s_1 \xrightarrow{a_2} s_2 \xrightarrow{a_4} s_1 \xrightarrow{a_2} s_2 \xrightarrow{a_3} s_5 \xrightarrow{a_1} \cdots$$
>
> ---

| 访问次序 | 状态–动作对 | First-Visit | Every-Visit |
|:---:|:---:|:---:|:---:|
| 1 | $(s_1,\ a_2)$ | ✅ 用（本回合首次） | ✅ 用 |
| 2 | $(s_2,\ a_4)$ | ✅ 用 | ✅ 用 |
| 3 | $(s_1,\ a_2)$ | ❌ 跳过（本回合重复） | ✅ 用 |
| 4 | $(s_2,\ a_3)$ | ✅ 用 | ✅ 用 |
| 5 | $(s_5,\ a_1)$ | ✅ 用 | ✅ 用 |

> ---
>
> **本 Notebook 对照**：第三节用**均值累计法**实现 Every-Visit；第四节用**覆盖写入法**实现 First-Visit——逆序遍历时，同一 $(s,a)$ 靠前的访问会覆盖靠后的，最终 Q 表里留下的是**首次访问对应的折扣回报**。

## 一、导入依赖库

In [1]:
import numpy as np       # 导入 NumPy 库，用于数值计算和矩阵运算，版本要求 >=1.18
import random            # 导入 Python 标准库 random，用于随机数生成
import importlib.util    # 导入 importlib.util，用于按文件路径动态加载模块

# 文件名 "02.1.ModelFree_Env_GridWorldV2.py" 以数字开头且含点号，不符合 Python 标识符规则，无法直接 import
# spec_from_file_location：根据给定模块别名和 .py 文件路径创建模块规格，返回 ModuleSpec 对象
_spec = importlib.util.spec_from_file_location("GridWorld_v2", "02.1.ModelFree_Env_GridWorldV2.py")
# module_from_spec：根据模块规格创建模块对象，此时模块代码尚未执行，返回 module 对象
GridWorld_v2 = importlib.util.module_from_spec(_spec)
# exec_module：执行模块代码完成初始化，之后可通过 GridWorld_v2.GridWorld_v2(...) 正常使用类
_spec.loader.exec_module(GridWorld_v2)

## 二、初始化网格世界与策略

In [2]:
gamma = 0.9   # 折扣因子 γ，float，控制未来奖励的衰减比例，越小越重视即时奖励

rows = 5      # 网格世界的行数，int，需与 desc 描述字符串的行数一致
columns = 5   # 网格世界的列数，int，需与 desc 描述字符串每行字符数一致

# 使用描述字符串初始化网格世界：'.' 表示普通格，'#' 表示禁止区（得分 -10），'T' 表示目标格（得分 1）
gridworld = GridWorld_v2.GridWorld_v2(
    forbiddenAreaScore=-10,  # 禁止区域的即时奖励，float，负值表示惩罚
    score=1,                 # 目标区域的即时奖励，float
    desc=[".....", ".##..", "..#..", ".#T#.", ".#..."]  # 网格布局描述，list[str]，共 5 行 5 列
)
gridworld.show()  # 以 emoji 打印网格世界布局，无返回值

value = np.zeros(rows * columns)        # 状态价值函数 V(s) 初始化，np.ndarray，shape=(25,)，全零
qtable = np.zeros((rows * columns, 5))  # 动作价值函数 Q(s,a) 初始化，np.ndarray，shape=(25, 5)，全零

# 随机初始化确定性策略，利用 NumPy 花式索引（Fancy Indexing）一步完成整数索引→one-hot 转换：
# np.random.randint(0,5,size=(rows*columns))：生成 25 个随机动作索引，shape=(25,)，每个值∈[0,4]，int
# np.eye(5)[整数数组]：花式索引——对数组中每个整数 i，取 np.eye(5) 第 i 行（动作 i 的 one-hot 向量）
#   并沿第 0 轴堆叠，等价于 np.stack([np.eye(5)[i] for i in idx])，但由 C 实现、效率更高
# 结果 policy：np.ndarray，shape=(25, 5)，每行恰有一个 1（对应该状态随机选定的动作），其余为 0
policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]
gridworld.showPolicy(policy)  # 可视化初始随机策略，无返回值


⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
➡️🔄⬇️⬆️⬅️
⬆️🔄🔄⬇️⬅️
➡️⬇️⏬⬇️⬆️
🔄🔄✅⏪⬅️
➡️🔄⬇️🔄⬆️


## 三、Every-Visit MC 策略迭代

In [3]:
# Every-Visit MC：同一状态-动作对在一条轨迹中多次出现时，每次出现的回报都参与均值估计
policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]  # 重新随机初始化策略，shape=(25, 5)，one-hot 编码
gridworld.show()              # 打印网格世界布局
gridworld.showPolicy(policy)  # 打印初始随机策略
print("初始随机策略（迭代前）")  # 提示当前使用随机策略

trajectorySteps = 100         # 每条轨迹的最大采样步数，int
qtable = np.zeros((rows * columns, 5))  # 初始化 Q 表，np.ndarray，shape=(25, 5)，全零
qtable_pre = qtable.copy() + 1          # 保存上一轮 Q 表快照（初始加 1 保证首轮进入循环），shape=(25, 5)

iteration = 0  # 迭代计数器，int，记录当前策略迭代轮次，初始为 0

# 当两轮 Q 表元素差的平方和大于 0.001 时，继续策略评估与改进
while np.sum((qtable_pre - qtable) ** 2) > 0.001:
    iteration += 1  # 每进入一轮迭代，计数器加 1
    print(f"\n{'=' * 50}")  # 打印分隔线，标识新一轮迭代的开始
    print(f"  第 {iteration} 轮迭代  |  迭代开始 Q 差异：{np.sum((qtable_pre - qtable) ** 2):.6f}")
    print(f"{'=' * 50}")  # 打印分隔线下边框
    qtable_pre = qtable.copy()  # 保存当前 Q 表为上一轮快照，np.ndarray，shape=(25, 5)

    # [Exploring Starts] 双重循环穷举所有 rows*columns*5 = 125 个 (s,a) 对，
    # 每对都将作为 getTrajectoryScore 的强制起点（nowState=i, action=j），
    # 保证每个 (s,a) 在每轮策略评估中至少作为一条 episode 的起点被采样一次，
    # 从而覆盖全部状态-动作空间，避免 Q(s,a) 因从未被访问而无法更新
    for i in range(rows * columns):  # [Exploring Starts] 外层循环：穷举所有起始状态 s，i 为状态编号，int，范围 [0, rows*columns=25)
        for j in range(5):           # [Exploring Starts] 内层循环：穷举所有起始动作 a，j 为动作编号，int，范围 [0, 5)
            # 初始化当前 episode 的状态-动作对累计回报，list[list[float]]，shape=(25, 5)，全零
            qtable_rewards = [[0 for _ in range(5)] for _ in range(rows * columns)]
            # 初始化当前 episode 的状态-动作对访问次数，list[list[int]]，shape=(25, 5)，全零
            qtable_nums = [[0 for _ in range(5)] for _ in range(rows * columns)]

            # 从状态 i 出发执行动作 j，按策略 policy 采样 trajectorySteps 步轨迹
            # 返回值：list[tuple]，长度为 trajectorySteps+1，每个元组为 (nowState, nowAction, score, nextState, nextAction)
            Trajectory = gridworld.getTrajectoryScore(
                nowState=i, action=j, policy=policy, steps=trajectorySteps
            )

            # Trajectory 共 trajectorySteps+1 个元素（索引 0..trajectorySteps），
            # 每个元素为 5 元组 (nowState, nowAction, score, nextState, nextAction)：
            #   nowState   : int，当前步所在状态编号，范围 [0, rows*columns)
            #   nowAction  : int，当前步执行的动作编号，范围 [0, 5)
            #   score      : float，当前步获得的即时奖励 r_t
            #   nextState  : int，执行动作后到达的下一状态编号
            #   nextAction : int，下一状态按策略选定的动作编号（供 SARSA 等算法使用）
            # 取索引 trajectorySteps 处（即最后一步，步号 T = trajectorySteps）的元组，
            # 只需即时奖励 r_T；其余字段（nowState/nowAction/nextState/nextAction）
            # 在逆向递推的起点初始化阶段无用，故用 _ 忽略。
            # 令 G = r_T 作为逆向折扣回报递推的初始值：终止时刻之后无后续奖励，
            # 因此 G_T = r_T + γ·0 = r_T，后续每步按 G_k = r_k + γ·G_{k+1} 递推
            _, _, score, _, _ = Trajectory[trajectorySteps]

            # 从倒数第二步向前逆向遍历，利用 G_t = r_t + γ*G_{t+1} 递推折扣累计回报
            for k in range(trajectorySteps - 1, -1, -1):
                tmpstate, tmpaction, tmpscore, _, _ = Trajectory[k]  # 解包第 k 步：状态、动作、即时奖励
                score = score * gamma + tmpscore      # 更新折扣累计回报 G_k = r_k + γ*G_{k+1}，float
                qtable_rewards[tmpstate][tmpaction] += score  # 累加该状态-动作对的折扣回报，float
                qtable_nums[tmpstate][tmpaction] += 1         # 该状态-动作对的访问次数加 1，int
                # Every-Visit：每次访问都参与均值，Q(s,a) ← 累计回报之和 / 访问次数
                qtable[tmpstate][tmpaction] = (
                    qtable_rewards[tmpstate][tmpaction] / qtable_nums[tmpstate][tmpaction]
                )

    # 策略改进：取每个状态 Q 值最大的动作构造新的 one-hot 确定性策略，shape=(25, 5)
    policy = np.eye(5)[np.argmax(qtable, axis=1)]
    q_diff = np.sum((qtable_pre - qtable) ** 2)  # 计算本轮 Q 表变化量（平方和），float
    print(f"  更新后策略：")  # 打印策略可视化标签
    gridworld.showPolicy(policy)           # 可视化当前更新后的策略
    print(f"  Q 均值：{qtable.mean():.6f}  |  本轮 Q 差异：{q_diff:.6f}  {'（已收敛）' if q_diff <= 0.001 else '（继续迭代）'}")

print(f"\n{'*' * 50}")  # 打印最终分隔线
print(f"  Every-Visit MC 迭代完成，共迭代 {iteration} 轮")  # 打印迭代总轮次
print(f"{'*' * 50}")


⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
🔄⬆️⬆️⬅️🔄
⬅️⏪⏫️⬇️➡️
⬆️⬆️⏬⬅️🔄
⬇️⏬✅⏩️⬆️
➡️⏪➡️➡️➡️
初始随机策略（迭代前）

  第 1 轮迭代  |  迭代开始 Q 差异：125.000000
  更新后策略：
🔄⬅️➡️➡️🔄
⬆️⏪⏫️⬆️⬆️
🔄⬅️⏪➡️⬇️
⬆️⏫️✅⏩️⬆️
⬆️⏩️🔄⬅️⬆️
  Q 均值：-20.361183  |  本轮 Q 差异：94235.535842  （继续迭代）

  第 2 轮迭代  |  迭代开始 Q 差异：94235.535842
  更新后策略：
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️➡️⬆️
  Q 均值：-2.440000  |  本轮 Q 差异：77810.484280  （继续迭代）

  第 3 轮迭代  |  迭代开始 Q 差异：77810.484280
  更新后策略：
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️⬅️⬆️
  Q 均值：-0.322669  |  本轮 Q 差异：2338.731022  （继续迭代）

  第 4 轮迭代  |  迭代开始 Q 差异：2338.731022
  更新后策略：
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️⬅️⬅️
  Q 均值：0.001253  |  本轮 Q 差异：328.030710  （继续迭代）

  第 5 轮迭代  |  迭代开始 Q 差异：328.030710
  更新后策略：
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
  Q 均值：0.292773  |  本轮 Q 差异：265.703149  （继续迭代）

  第 6 轮迭代  |  迭代开始 Q 差异：265.703149
  更新后策略：
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬇️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
  Q 均值：0.555204  |  本轮 Q 差异：215.217921  （继续迭代）

  第 7 轮迭代  |  迭代开始

## 四、First-Visit MC 策略迭代

In [11]:
# First-Visit MC：同一状态-动作对在一条轨迹中多次出现时，只取最后一次出现的回报（覆盖写入 Q 表）
policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]  # 重新随机初始化策略，shape=(25, 5)，one-hot 编码
gridworld.show()              # 打印网格世界布局
gridworld.showPolicy(policy)  # 打印初始随机策略
print("初始随机策略（迭代前）")  # 提示当前使用随机策略

trajectorySteps = 100         # 每条轨迹的最大采样步数，int
qtable = np.zeros((rows * columns, 5))  # 初始化 Q 表，np.ndarray，shape=(25, 5)，全零
qtable_pre = qtable.copy() + 1          # 保存上一轮 Q 表快照，shape=(25, 5)，初始加 1 确保进入循环

iteration = 0  # 迭代计数器，int，记录当前策略迭代轮次，初始为 0

while np.sum((qtable_pre - qtable) ** 2) > 0.001:
    iteration += 1  # 每进入一轮迭代，计数器加 1
    print(f"\n{'=' * 50}")  # 打印分隔线，标识新一轮迭代的开始
    print(f"  第 {iteration} 轮迭代  |  迭代开始 Q 差异：{np.sum((qtable_pre - qtable) ** 2):.6f}")
    print(f"{'=' * 50}")  # 打印分隔线下边框
    qtable_pre = qtable.copy()  # 保存当前 Q 表为上一轮快照，np.ndarray，shape=(25, 5)

    # [Exploring Starts] 双重循环穷举所有 rows*columns*5 = 125 个 (s,a) 对，
    # 每对都将作为 getTrajectoryScore 的强制起点（nowState=i, action=j），
    # 保证每个 (s,a) 在每轮策略评估中至少作为一条 episode 的起点被采样一次，
    # 从而覆盖全部状态-动作空间，避免 Q(s,a) 因从未被访问而无法更新
    for i in range(rows * columns):  # [Exploring Starts] 外层循环：穷举所有起始状态 s，i 为状态编号，int，范围 [0, rows*columns=25)
        for j in range(5):           # [Exploring Starts] 内层循环：穷举所有起始动作 a，j 为动作编号，int，范围 [0, 5)
            # 从状态 i 出发执行动作 j，按策略 policy 采样 trajectorySteps 步轨迹
            # 返回值：list[tuple]，长度为 trajectorySteps+1
            Trajectory = gridworld.getTrajectoryScore(
                nowState=i, action=j, policy=policy, steps=trajectorySteps
            )

            # 取轨迹末尾一步的即时奖励作为折扣回报递推的起点，float
            _, _, score, _, _ = Trajectory[trajectorySteps]

            # 从倒数第二步向前逆向遍历，递推计算折扣累计回报
            for k in range(trajectorySteps - 1, -1, -1):
                tmpstate, tmpaction, tmpscore, _, _ = Trajectory[k]  # 解包第 k 步：状态、动作、即时奖励
                score = score * gamma + tmpscore  # G_k = r_k + γ*G_{k+1}，float
                # First-Visit（覆盖写入）：直接用当前折扣回报覆盖 Q 值，后续出现会继续覆盖，相当于保留最后一次
                qtable[tmpstate][tmpaction] = score  # Q(s,a) ← G，float

    print(f"  状态 0 Q值：{qtable[0]}")  # 打印状态 0 的 Q 值向量，np.ndarray，shape=(5,)，用于调试
    print(f"  状态 1 Q值：{qtable[1]}")  # 打印状态 1 的 Q 值向量，np.ndarray，shape=(5,)，用于调试

    # 策略改进：取每个状态 Q 值最大的动作构造新的 one-hot 确定性策略，shape=(25, 5)
    policy = np.eye(5)[np.argmax(qtable, axis=1)]
    q_diff = np.sum((qtable_pre - qtable) ** 2)  # 计算本轮 Q 表变化量（平方和），float
    print(f"  更新后策略：")  # 打印策略可视化标签
    gridworld.showPolicy(policy)  # 可视化当前更新后的策略
    print(f"  本轮 Q 差异：{q_diff:.6f}  {'（已收敛）' if q_diff <= 0.001 else '（继续迭代）'}")

print(f"\n{'*' * 50}")  # 打印最终分隔线
print(f"  First-Visit MC 迭代完成，共迭代 {iteration} 轮")  # 打印迭代总轮次
print(f"{'*' * 50}")

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
⬆️⬇️⬆️🔄🔄
🔄⏩️⏫️⬇️⬇️
➡️⬇️⏬➡️⬇️
⬆️⏪✅⏩️➡️
🔄⏩️⬅️⬅️🔄
random policy
125.0
qtable[0]: [ -9.99973439 -23.66076095   0.          -9.99976095  -8.99976095]
qtable[1]: [-24.66076095  -8.99976095 -26.28973439  -8.99976095 -23.66076095]
⬇️➡️➡️➡️⬅️
🔄⏪⏬⬆️⬆️
⬆️➡️⏬⬅️⬆️
⬇️⏩️✅⏪⬇️
🔄⏪⬆️➡️🔄
58724.65101582276
qtable[0]: [-1.  0.  0. -1.  0.]
qtable[1]: [ -1.   0. -10.   0.   0.]
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️⬅️⬆️
54289.483682466176
qtable[0]: [-1.  0.  0. -1.  0.]
qtable[1]: [ -1.   0. -10.   0.   0.]
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️⬅️⬅️
668.2177293353504
qtable[0]: [-1.  0.  0. -1.  0.]
qtable[1]: [ -1.   0. -10.   0.   0.]
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
265.70307335590013
qtable[0]: [-1.  0.  0. -1.  0.]
qtable[1]: [ -1.   0. -10.   0.   0.]
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬇️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
215.21792104600723
qtable[0]: [-1.  0.  0. -1.  0.]
qtable[1]: [ -1.   0. -10.   0.   0.]
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬇️
⬆️⬅️⏬